# Audit des données

Objectif : vérifier la disponibilité et la qualité des données
avant de comparer les portefeuilles.

Première étape : collecte pilote de SPY, IEF, GLD et BIL
sur la période 2019–2023.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

project_dir = Path.cwd().resolve()

if project_dir.name == "notebooks":
    project_dir = project_dir.parent

if not (project_dir / "config.py").exists():
    raise FileNotFoundError("Le dossier du projet n'a pas été retrouvé.")

if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from config import (
    TICKERS,
    PILOT_START,
    PILOT_END,
    PILOT_DOWNLOAD_END,
    RAW_DATA_DIR,
)
from src.data_collection import download_etfs, save_data, load_data

print("Projet :", project_dir)
print("ETF étudiés :", TICKERS)

Projet : C:\Users\DHERALE\Desktop\Projet-Python-ENSAE
ETF étudiés : ['SPY', 'IEF', 'GLD', 'BIL']


In [ ]:
# un mois supplémentaire pour calculer le rendement de janvier
download_start = (
    pd.Timestamp(PILOT_START) - pd.DateOffset(months=1)
).strftime("%Y-%m-%d")

data_folder = RAW_DATA_DIR / "pilote_2019_2023"

if not (data_folder / "metadata.json").exists():
    datasets = download_etfs(
        TICKERS,
        download_start,
        PILOT_DOWNLOAD_END,
    )

    save_data(
        datasets,
        data_folder,
        download_start,
        PILOT_DOWNLOAD_END,
    )

    print("Collecte terminée et sauvegardée.")
else:
    print("Utilisation des fichiers déjà sauvegardés.")

# on repart toujours des fichiers pour réaliser l'audit
raw_data, metadata = load_data(data_folder)

# vérif des fichiers 
if (
    set(metadata["tickers"]) != set(TICKERS)
    or metadata["start"] != download_start
    or metadata["end_exclusive"] != PILOT_DOWNLOAD_END
):
    raise ValueError("Les fichiers ne correspondent pas à la collecte demandée.")

print("Sauvegarde du :", metadata["saved_at_utc"])

for ticker, data in raw_data.items():
    print(f"{ticker} : {len(data)} lignes")

Téléchargement de SPY...
Téléchargement de IEF...
Téléchargement de GLD...
Téléchargement de BIL...
Collecte terminée et sauvegardée.
Sauvegarde du : 2026-10-23T12:46:50.695516+00:00
SPY : 1277 lignes
IEF : 1277 lignes
GLD : 1277 lignes
BIL : 1277 lignes


In [5]:
# on sélectionne la période étudiée
pilot_data = {}

for ticker, data in raw_data.items():
    pilot_data[ticker] = data.loc[PILOT_START:PILOT_END].copy()

# diagnostic par ETF
results = []

for ticker, data in pilot_data.items():
    prices = data["Adj Close"]
    valid_prices = prices.dropna()

    results.append({
        "ETF": ticker,
        "premier_prix_valide": valid_prices.index.min(),
        "dernier_prix_valide": valid_prices.index.max(),
        "nombre_lignes": len(data),
        "prix_manquants": int(prices.isna().sum()),
        "dates_dupliquees": int(data.index.duplicated().sum()),
        "dates_triees": data.index.is_monotonic_increasing,
        "prix_non_positifs": int((prices <= 0).sum()),
        "distributions": int(data["Dividends"].fillna(0).ne(0).sum()),
    })

quality_table = pd.DataFrame(results).set_index("ETF")
display(quality_table)

columns_to_check = [
    "Close", "Adj Close", "Dividends", "Stock Splits"
]

missing_values = {}

for ticker, data in pilot_data.items():
    missing_values[ticker] = data[columns_to_check].isna().sum()

display(pd.DataFrame(missing_values).T)

,premier_prix_valide,dernier_prix_valide,nombre_lignes,prix_manquants,dates_dupliquees,dates_triees,prix_non_positifs,distributions
ETF,,,,,,,,
SPY,2019-01-02,2023-12-29,1258,0,0,True,0,20
IEF,2019-01-02,2023-12-29,1258,0,0,True,0,60
GLD,2019-01-02,2023-12-29,1258,0,0,True,0,0
BIL,2019-01-02,2023-12-29,1258,0,0,True,0,38


,Close,Adj Close,Dividends,Stock Splits
SPY,0,0,0,0
IEF,0,0,0,0
GLD,0,0,0,0
BIL,0,0,0,0


In [4]:
# étude doublons
if quality_table["dates_dupliquees"].sum() > 0:
    raise ValueError("Des dates sont dupliquées dans les données.")

# on réunit les prix ajustés pour les quatre ETF dans un Dataframe
price_series = {}

for ticker, data in pilot_data.items():
    price_series[ticker] = data["Adj Close"]

adjusted_prices = pd.DataFrame(price_series).sort_index()

# on repère les dates où au moins un ETF n'a pas de prix valide.
missing_dates = adjusted_prices[
    adjusted_prices.isna().any(axis=1)
]

print("Dates présentes pour au moins un ETF :", len(adjusted_prices))
print("Dates avec quatre prix valides :", len(adjusted_prices.dropna()))
print("Dates avec un prix manquant :", len(missing_dates))

display(missing_dates.head(20))

Dates présentes pour au moins un ETF : 1258
Dates avec quatre prix valides : 1258
Dates avec un prix manquant : 0


,SPY,IEF,GLD,BIL
Date,,,,
